In [1]:
# setup pyspark environment
import os
import sys

if sys.platform == "darwin":
    os.environ["SPARK_HOME"] = "/Users/hubert/Documents/dev/python/data-analysis-pyspark/spark-4.0.0-bin-hadoop3"
    os.environ["PATH"] += os.pathsep + "/Users/hubert/Documents/dev/python/data-analysis-pyspark/spark-4.0.0-bin-hadoop3/bin"

In [7]:
%pip install pandas
%pip install pyarrow


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.3/34.3 MB 43.2 MB/s  0:00:00 eta 0:00:01

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("MyApp") \
    .config("spark.driver.extraJavaOptions", "-Dio.netty.tryReflectionSetAccessible=true") \
    .config("spark.executor.extraJavaOptions", "-Dio.netty.tryReflectionSetAccessible=true") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/08 20:21:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [8]:
import pyspark.sql.functions as F 
import pandas as pd

df = spark.createDataFrame(pd.DataFrame({'hi': ['hello', 'hi', 'hey']}))

df.select(F.upper(F.col('hi'))).show()

+---------+
|upper(hi)|
+---------+
|    HELLO|
|       HI|
|      HEY|
+---------+



In [12]:
import pyspark.sql.types as T
df2 = spark.createDataFrame(pd.DataFrame({'temps': [32.0, 98.0, 68.0]}), schema=T.StructType([T.StructField("temps", T.DoubleType(), True)]))

@F.pandas_udf(T.DoubleType())
def f_to_c(temps: pd.Series) -> pd.Series:
    return (temps - 32) * 5.0/9.0 

df2.select(f_to_c(F.col('temps')).alias('temps_celsius')).show()

df2.show()

df2.withColumn('temps_celsius', f_to_c(F.col('temps'))).show()

df2.show()

+------------------+
|     temps_celsius|
+------------------+
|               0.0|
|36.666666666666664|
|              20.0|
+------------------+

+-----+
|temps|
+-----+
| 32.0|
| 98.0|
| 68.0|
+-----+

+-----+------------------+
|temps|     temps_celsius|
+-----+------------------+
| 32.0|               0.0|
| 98.0|36.666666666666664|
| 68.0|              20.0|
+-----+------------------+

+-----+
|temps|
+-----+
| 32.0|
| 98.0|
| 68.0|
+-----+

